In [11]:
%pwd

'c:\\Users\\dipjy\\OneDrive\\Desktop\\Career_2026\\Projects\\Mlops_projects\\Predictive_maintanance_system'

In [12]:
from dataclasses import dataclass
from typing import List
from pathlib import Path


@dataclass
class FeatureEngineeringConfig:
    root_dir: str
    master_data_path: str
    feature_store_path: str
    final_feature_path: str
    lag_features: list
    rolling_windows: list
    prediction_horizons: list
    telemetry_columns: list
    machine_id_column: str
    datetime_column: str
    failure_column: str    

In [13]:
from dataclasses import dataclass

@dataclass
class FeatureEngineeringArtifact:
    feature_store_path: str
    final_feature_path: str
    is_engineering_successful: bool
    message: str  

In [14]:
from src.utils.main_utils import read_yaml, create_directories
from src.constants import CONFIG_FILE_PATH, SCHEMA_FILE_PATH
from pathlib import Path

class ConfigurationManager:
    
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        create_directories([
            self.config.artifacts_root,
            self.config.data_validation.root_dir
        ])

    def get_feature_engineering_config(self) -> FeatureEngineeringConfig:
        config = self.config.feature_engineering

        create_directories([config.root_dir])

        return FeatureEngineeringConfig(
            root_dir=config.root_dir,
            master_data_path=config.master_data_path,
            feature_store_path=config.feature_store_path,
            final_feature_path=config.final_feature_path,

            lag_features=config.lag_features,
            rolling_windows=config.rolling_windows,
            prediction_horizons=config.prediction_horizons,

            telemetry_columns=config.telemetry_columns,

            machine_id_column=config.machine_id_column,
            datetime_column=config.datetime_column,
            failure_column=config.failure_column
        )

In [15]:
from src.components.feature_engineering import FeatureEngineering

feature_engineering_config = ConfigurationManager().get_feature_engineering_config()
feature_engineering = FeatureEngineering(config=feature_engineering_config)

[ 2026-05-07 19:45:32,643 ] root - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-05-07 19:45:32,735 ] root - INFO - yaml file: config\schema.yaml loaded successfully
[ 2026-05-07 19:45:32,767 ] root - INFO - created directory at: artifacts
[ 2026-05-07 19:45:32,779 ] root - INFO - created directory at: artifacts/data_validation
[ 2026-05-07 19:45:32,817 ] root - INFO - created directory at: artifacts/feature_engineering
[ 2026-05-07 19:45:32,852 ] root - INFO - yaml file: config\schema.yaml loaded successfully


In [16]:


class TrainingPipeline:
    def __init__(self):
        self.config_manager = ConfigurationManager()

    def start_feature_engineering(self):
        """
        Build the FeatureEngineering component and run the full feature engineering flow.
        """
        feature_engineering_config = self.config_manager.get_feature_engineering_config()
        feature_engineering = FeatureEngineering(config=feature_engineering_config)
        return feature_engineering.initiate_feature_engineering()

In [17]:
artifact = feature_engineering.initiate_feature_engineering()
print(artifact)

import pandas as pd

engineered_df = pd.read_csv(feature_engineering_config.final_feature_path)
print(engineered_df.shape)
engineered_df.head()

[ 2026-05-07 19:57:25,585 ] root - INFO - Starting feature engineering...
[ 2026-05-07 20:11:09,632 ] root - INFO - Feature engineering completed successfully.


FeatureEngineeringArtifact(feature_store_path='artifacts/feature_engineering/feature_store.csv', final_feature_path='artifacts/feature_engineering/final_feature_dataset.csv', is_engineering_successful=True, message='Feature engineering completed successfully')
(784181, 84)


,datetime,machineID,volt,rotate,pressure,vibration,model,age,failure_comp1,failure_comp2,...,vibration_delta_3,time_since_last_maintenance,maintenance_count_last_30_days,component_replaced_last_30_days,error_count_last_24h,error_count_last_7d,recent_error_type,prior_failure_count,time_since_last_failure,failure_within_next_24h
0,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0.0,0.0,...,2.488450,0.0,1.0,1.0,0.0,3.0,errorID_error5,0.0,0.0,0.0
1,2015-01-05 07:00:00,1,155.511452,498.398435,103.068134,33.270415,model3,18,0.0,0.0,...,-11.931932,1.0,1.0,1.0,0.0,3.0,errorID_error5,1.0,1.0,0.0
2,2015-01-05 08:00:00,1,172.439821,392.124959,108.135159,39.477497,model3,18,0.0,0.0,...,-20.099753,2.0,1.0,1.0,0.0,3.0,errorID_error5,1.0,2.0,0.0
3,2015-01-05 09:00:00,1,138.826437,451.348967,126.464580,38.257462,model3,18,0.0,0.0,...,-14.125635,3.0,1.0,1.0,0.0,3.0,errorID_error5,1.0,3.0,0.0
4,2015-01-05 10:00:00,1,177.278402,403.199389,100.858613,38.776021,model3,18,0.0,0.0,...,5.505606,4.0,1.0,1.0,0.0,3.0,errorID_error5,1.0,4.0,0.0
